In [1]:
# Cell 0: cek semua input dan salin script pelatihan
# Input yang perlu ditambahkan (Add Input): notebook yang berisi 11_train_student_lat.py (mis. notebook 23),
# data v1 (_meta.csv), report_labels.csv (notebook 01d), knee_side.csv (notebook 22), oof_effb0_kf2.csv (guru versi 2)
import os, shutil

def find_file(name):
    hasil = []
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d != "competitions"]
        if name in files:
            hasil.append(os.path.join(root, name))
    return hasil

for f in ["11_train_student_lat.py", "_meta.csv", "report_labels.csv", "knee_side.csv", "oof_effb0_kf2.csv"]:
    print(f"{f:25s} -> {find_file(f)}")

shutil.copy(find_file("11_train_student_lat.py")[0], "/kaggle/working/11_train_student_lat.py")
print("\nScript disalin.")

11_train_student_lat.py   -> ['/kaggle/input/notebooks/kellenvalerie/rsna-knee-23-dinov2-lat/11_train_student_lat.py']
_meta.csv                 -> ['/kaggle/input/notebooks/kellenvalerie/rsna-knee-02-preprocess-mri/vol_train/_meta.csv']
report_labels.csv         -> ['/kaggle/input/notebooks/kellenvalerie/rsna-knee-01d-label-ensemble/report_labels.csv']
knee_side.csv             -> ['/kaggle/input/notebooks/kellenvalerie/rsna-knee-22-sisi-lutut/knee_side.csv']
oof_effb0_kf2.csv         -> ['/kaggle/input/notebooks/kellenvalerie/rsna-knee-06-tahapc-teacher-oof/oof_effb0_kf2.csv']

Script disalin.


In [2]:
%%bash
# Cell 1: uji coba singkat EfficientNet-B0 dengan guru (sekitar 3 menit)
# Pastikan muncul baris "Label diperbaiki: alpha=0.5 (guru: oof_effb0_kf2.csv)" dan "Normalisasi sisi lutut"
cd /kaggle/working
python -u 11_train_student_lat.py --alpha 0.5 --oof oof_effb0_kf2.csv --meta _meta.csv \
    --backbone efficientnet_b0 --lr_backbone 1e-4 --batch 4 --epochs 12 \
    --side_csv knee_side.csv --seed 42 --tag uji --debug 2>&1 | grep -v Warning | tail -n 22

Label diperbaiki: alpha=0.5 (guru: oof_effb0_kf2.csv)
Train 64 | Val 32 | Gold holdout 58
Jumlah irisan per view: 16 | ukuran: 224 px | batch 4 x accum 1
Jumlah view per studi: 3
Backbone: efficientnet_b0 | tag: uji | parameter: 4.5 juta
  ep0 it0/16 loss 0.6994 (44.89 s/it)
Epoch 0: loss 0.5789 | val AUC 0.6114 | val-ref AUC 0.5833 | gold AUC 0.5612 | 1.0 menit

=== [uji] Checkpoint terbaik (val-ref AUC 0.5833) pada label emas ===
ACL                AUC=0.471
MCL                AUC=0.437
Medial Meniscus    AUC=0.618
Lateral Meniscus   AUC=0.461
Medial OA          AUC=0.560
Lateral OA         AUC=0.523
PF OA              AUC=0.595
Effusion           AUC=0.686
Synovitis          AUC=0.570
Baker's            AUC=0.522
Contusion          AUC=0.607
Fracture           AUC=0.685
Mean AUC gold: 0.5612


In [3]:
%%bash
# Cell 2: pelatihan penuh, sekitar 4,5 jam (dua GPU berjalan bersamaan)
#   GPU 0: DINOv2-S lat DENGAN guru seed 11 (ablasi seed ke-3), lalu EffB0 lat seed 42 (perbandingan adil)
#   GPU 1: DINOv2-S lat TANPA guru seed 11 (ablasi seed ke-3), lalu EffB0 lat seed 7  (perbandingan adil)
# Semua model memakai data v1 + lat, label perbaikan yang sama, 12 epoch, LR backbone 1e-4, batch 4.
cd /kaggle/working
rm -f uji_* train_log_uji.csv val_preds_uji.csv gold_preds_uji.csv
DATA="--meta _meta.csv --side_csv knee_side.csv --batch 4 --epochs 12 --lr_backbone 1e-4"
GURU="--alpha 0.5 --oof oof_effb0_kf2.csv"
DINO="--backbone vit_small_patch14_dinov2.lvd142m --llrd 0.8"
EFF="--backbone efficientnet_b0"

( CUDA_VISIBLE_DEVICES=0 python -u 11_train_student_lat.py $DATA $GURU $DINO --seed 11 \
      --tag dinov2s_student_v1d_lat_s11 > log_dinov2s_student_v1d_lat_s11.txt 2>&1
  CUDA_VISIBLE_DEVICES=0 python -u 11_train_student_lat.py $DATA $GURU $EFF --seed 42 \
      --tag effb0_student_v1d_lat > log_effb0_student_v1d_lat.txt 2>&1 ) &

( CUDA_VISIBLE_DEVICES=1 python -u 11_train_student_lat.py $DATA --alpha 1.0 $DINO --seed 11 \
      --tag dinov2s_v1d_lat_noteacher_s11 > log_dinov2s_v1d_lat_noteacher_s11.txt 2>&1
  CUDA_VISIBLE_DEVICES=1 python -u 11_train_student_lat.py $DATA $GURU $EFF --seed 7 \
      --tag effb0_student_v1d_lat_s7 > log_effb0_student_v1d_lat_s7.txt 2>&1 ) &
wait
for t in dinov2s_student_v1d_lat_s11 dinov2s_v1d_lat_noteacher_s11 effb0_student_v1d_lat effb0_student_v1d_lat_s7; do
  echo "===== $t ====="; tail -n 16 log_$t.txt
done
grep -H "Label diperbaiki\|Normalisasi\|Train \|LLRD\|Error\|error" log_*.txt

===== dinov2s_student_v1d_lat_s11 =====
Epoch 11: loss 0.3751 | val AUC 0.8738 | val-ref AUC 0.8917 | gold AUC 0.8679 | 14.9 menit

=== [dinov2s_student_v1d_lat_s11] Checkpoint terbaik (val-ref AUC 0.8930) pada label emas ===
ACL                AUC=0.930
MCL                AUC=0.918
Medial Meniscus    AUC=0.887
Lateral Meniscus   AUC=0.768
Medial OA          AUC=0.971
Lateral OA         AUC=0.838
PF OA              AUC=0.758
Effusion           AUC=0.912
Synovitis          AUC=0.706
Baker's            AUC=0.953
Contusion          AUC=0.899
Fracture           AUC=0.904
Mean AUC gold: 0.8703
===== dinov2s_v1d_lat_noteacher_s11 =====
Epoch 11: loss 0.3418 | val AUC 0.8456 | val-ref AUC 0.8187 | gold AUC 0.8634 | 14.8 menit

=== [dinov2s_v1d_lat_noteacher_s11] Checkpoint terbaik (val-ref AUC 0.8188) pada label emas ===
ACL                AUC=0.908
MCL                AUC=0.934
Medial Meniscus    AUC=0.899
Lateral Meniscus   AUC=0.719
Medial OA          AUC=0.972
Lateral OA         AUC=0.812


In [4]:
# Cell 3: ringkasan checkpoint terbaik setiap model
import pandas as pd
rows = []
for tag in ["dinov2s_student_v1d_lat_s11", "dinov2s_v1d_lat_noteacher_s11",
            "effb0_student_v1d_lat", "effb0_student_v1d_lat_s7"]:
    log = pd.read_csv(f"/kaggle/working/train_log_{tag}.csv")
    b = log.loc[log["val_ref_auc"].idxmax()]                   # aturan checkpoint sama dengan script
    rows.append({"model": tag, "epoch": int(b["epoch"]), "val": round(b["val_auc"], 4),
                 "gold": round(b["gold_auc"], 4), "menit/epoch": round(log["min"].mean(), 1)})
print(pd.DataFrame(rows).to_string(index=False))

                        model  epoch    val   gold  menit/epoch
  dinov2s_student_v1d_lat_s11      9 0.8754 0.8703         14.9
dinov2s_v1d_lat_noteacher_s11      8 0.8452 0.8576         14.8
        effb0_student_v1d_lat      9 0.8693 0.8570          7.2
     effb0_student_v1d_lat_s7      9 0.8649 0.8640          7.2
